In [1]:
import os
import numpy as np
import pandas as pd

# 1. CREATE DATA ARCHITECTURE FOLDER
os.makedirs('data', exist_ok=True)

# Set seed for exact reproducibility
np.random.seed(42)
n_readings = 1200

# 2. GENERATE CONTINUOUS PHYSICAL TELEMETRY
turbine_ids = [f"TURBINE-{np.random.randint(1, 6)}" for _ in range(n_readings)]
wind_speed_mps = np.random.uniform(0, 30, size=n_readings).round(2)
blade_pitch_deg = np.random.uniform(0, 45, size=n_readings).round(1)

# 3. EMBED REAL-WORLD AERODYNAMIC PHYSICS BOUNDARIES
# Kinetic power generation follows a cubic growth curve based on wind velocity
power_mw = 0.5 * 1.225 * (wind_speed_mps ** 3) * 0.45 / 10000
# Emergency Safety Cut-Out Threshold: High wind speed causes automatic shutdown (0 output)
power_mw = np.where(wind_speed_mps > 25, 0.0, power_mw)
# Cut-In Threshold: Wind is too weak to move the blades
power_mw = np.where(wind_speed_mps < 3.5, 0.0, power_mw)

# Mechanical Friction Constraint: Higher blade pitch angles degrade generation efficiency
power_mw = power_mw * (1 - (blade_pitch_deg * 0.015))
# Inject realistic environmental sensor noise
power_mw = np.clip(power_mw + np.random.normal(0, 0.1, size=n_readings), 0, 5.0).round(3)

# 4. CREATE THE DATAFRAME AND SAVE TO CSV
df_energy = pd.DataFrame({
    'TelemetryID': [f"TEL-{i:05d}" for i in range(1, n_readings + 1)],
    'TurbineID': turbine_ids,
    'WindSpeed_MPS': wind_speed_mps,
    'BladePitch_Deg': blade_pitch_deg,
    'PowerOutput_MW': power_mw
})

df_energy.to_csv('data/raw_turbine_telemetry.csv', index=False)
print("Project 4 IoT Telemetry Engine Active! Generated 'data/raw_turbine_telemetry.csv' with 1,200 sensor logs.")


Project 4 IoT Telemetry Engine Active! Generated 'data/raw_turbine_telemetry.csv' with 1,200 sensor logs.


In [2]:
# 1. LOAD THE TELEMETRY
df_grid = pd.read_csv('data/raw_turbine_telemetry.csv')

# 2. ISOLATE HARDWARE EMERGENCY STATE (Wind > 25 m/s causes zero power generation)
emergency_shutdowns = df_grid[(df_grid['WindSpeed_MPS'] > 25) & (df_grid['PowerOutput_MW'] == 0.0)]
active_generation = df_grid[(df_grid['WindSpeed_MPS'] >= 3.5) & (df_grid['WindSpeed_MPS'] <= 25)]

print("--- INDUSTRIAL OPERATIONS TELEMETRY REPORT ---")
print(f"Total Severe Weather Emergency Shutdowns Logged: {len(emergency_shutdowns)} events")
print(f"Maximum Power Output Recorded under standard operation: {df_grid['PowerOutput_MW'].max():.3f} MW\n")

# 3. MECHANICAL BLADE PITCH IMPACT CRITERIA
# Group by blade pitch to see how physical tilt degrades electrical efficiency
active_generation['Pitch_Bracket'] = pd.cut(active_generation['BladePitch_Deg'],
                                            bins=[0, 15, 30, 45],
                                            labels=['Optimal (0-15°)', 'Friction (16-30°)', 'Stalled (31-45°)'])

pitch_pivot = active_generation.groupby('Pitch_Bracket')['PowerOutput_MW'].mean()
print("--- TURBINE GENERATION EFFICIENCY LOSSES ---")
for bracket, avg_power in pitch_pivot.items():
    print(f"Blade Configuration [{bracket}]: Average Grid Output = {avg_power:.3f} MW")


--- INDUSTRIAL OPERATIONS TELEMETRY REPORT ---
Total Severe Weather Emergency Shutdowns Logged: 84 events
Maximum Power Output Recorded under standard operation: 0.576 MW

--- TURBINE GENERATION EFFICIENCY LOSSES ---
Blade Configuration [Optimal (0-15°)]: Average Grid Output = 0.119 MW
Blade Configuration [Friction (16-30°)]: Average Grid Output = 0.109 MW
Blade Configuration [Stalled (31-45°)]: Average Grid Output = 0.076 MW


/tmp/ipykernel_2755/40079314.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active_generation['Pitch_Bracket'] = pd.cut(active_generation['BladePitch_Deg'],
/tmp/ipykernel_2755/40079314.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pitch_pivot = active_generation.groupby('Pitch_Bracket')['PowerOutput_MW'].mean()


# Portfolio Project Phase 4: Industrial IoT Platform Engineering

## 📘 Code Explainer: Streaming Telemetry Physics Engine
*This document breaks down the physical properties and math logic used in the turbine sensor script for non-technical stakeholders.*

### 1. Simulating Physical Aerodynamics
* **`np.random.uniform(0, 30)`**: This simulates raw continuous data streaming from a turbine's anemometer wind velocity sensor, measuring wind speeds from completely dead air (0 m/s) to extreme storm gusts (30 m/s).
* **`power_mw = 0.5 * 1.225 * (wind_speed_mps ** 3) * 0.45 / 10000`**: This line implements core atmospheric physics. Kinetic wind power output increases as a cubic function of wind velocity, meaning small increases in wind create exponentially larger physical force on the turbine gears.
* **`np.where(wind_speed_mps > 25, 0.0, power_mw)`**: This models structural hardware emergency boundaries. If ambient wind speeds exceed 25 m/s, real-world operators immediately deploy the mechanical brakes, stopping rotation to prevent the internal generator from catching fire or spinning out of control.

### 2. Segmenting Mechanical Wear
* **`pd.cut(..., bins=bins, labels=labels)`**: This categorizes continuous blade pitch telemetry into three distinct physical operating postures (Optimal, Friction, and Stalled), helping platform engineers isolate exactly how hardware alignment angles destroy financial power grid output.

---

## 🤖 Autonomous AI Agent Architecture: The SCADA Grid Automation Controller
Instead of requiring human operators to manually monitor field sensors, we propose an autonomous **Supervisory Control and Data Acquisition (SCADA)** physical automation agent. This system uses our telemetry boundaries as its automated operational instructions.

